# Matrice co-esami per periodo (unica, normalizzata per anno)

Scarica **direttamente** il dump Esse3 da **resapp** (HTTP Basic, **in memoria**), estrae
gli esami **superati in un periodo** (date `YYMMDD`) e costruisce **un'unica matrice** di
co-esami, coi **codici così come sono** (LT e LM insieme).

- **Co-esame**: due esami formano una coppia **solo** se superati **dalla stessa matricola**
  **e** nello **stesso corso di studio** (entrambi LT, oppure entrambi LM). L'unità di
  conteggio è la **carriera = (matricola, CdS)**; le coppie si formano dentro ciascuna
  carriera, poi si aggrega tutto in una sola matrice (nessuna coppia LT↔LM dello stesso studente).
- **Normalizzazione per anno**: i conteggi vengono **divisi per la durata del periodo in
  anni** e **arrotondati all'intero** → `E` ed `N` sono **tassi annui** (interi),
  confrontabili a prescindere dalla lunghezza del periodo.
  `E` (diagonale) = carriere/anno che hanno superato il codice; `N` (fuori diagonale) =
  carriere/anno che hanno superato **entrambi**. Le celle che arrotondano a 0 restano **vuote**.
- **Equivalenze: ignorate** (i codici equivalenti li collassa la **web app**, via `canon`).
- **Spacchettamenti**: stessa matricola, **stesso giorno**, un codice prefisso di un altro
  (es. `204BB` e `204BB_`) → si tiene **solo il più corto**.

> ⚠️ **Privacy**: credenziali chieste a runtime e **mai salvate**; il dump resta **solo in
> memoria**. Si salva **solo** la matrice (aggregati per codice, nessun PII). Valori
> **interi** → nessun problema di locale all'import in gsheet.

In [ ]:
import os, io, base64, getpass, urllib.request, urllib.error
from itertools import combinations
from collections import Counter
import pandas as pd

# ---- PARAMETRI ----
URL = 'https://resapp.adm.unipi.it/powerbi_ext/fisica/FISICA_MONIT_CAR_TAB.csv'
SORGENTE_LOCALE = None   # path a un CSV gia' scaricato (test offline); None = scarica da resapp
START = '221001'   # inizio periodo, YYMMDD (incluso)
END   = '250930'   # fine periodo,   YYMMDD (incluso)
OUT_DIR = '.'      # dove salvare la matrice (qui: cartella matrice/)

# Nomi colonne nel dump (vedi CSV_SORGENTE.md)
COL_MAT  = 'MATRICOLA'
COL_COD  = 'COD'
COL_DATA = 'DATA_SUP'        # data di superamento (formato resapp: gg-mm-aaaa)
COL_TIPO = 'TIPO_CORSO_COD'  # 'L2' = triennale (LT), 'LM' = magistrale
TIPO_LT, TIPO_LM = 'L2', 'LM'

start_d = pd.to_datetime(START, format='%y%m%d')
end_d   = pd.to_datetime(END,   format='%y%m%d')
ANNI = (end_d - start_d).days / 365.25   # durata del periodo in anni (per la normalizzazione)
if ANNI <= 0:
    raise ValueError('Periodo non valido: END deve essere dopo START.')
print(f'Periodo: {start_d.date()} → {end_d.date()}  ({ANNI:.3f} anni)')

In [ ]:
# ---- DOWNLOAD DUMP (in memoria; credenziali mai salvate) ----
if SORGENTE_LOCALE:
    with open(SORGENTE_LOCALE, 'rb') as f:
        raw = f.read()
    print(f'letti {len(raw):,} byte dal file locale (test offline)')
else:
    utente = input('Utente resapp: ').strip()
    password = getpass.getpass('Password resapp: ')
    req = urllib.request.Request(URL)
    req.add_header('Authorization', 'Basic ' + base64.b64encode(f'{utente}:{password}'.encode()).decode())
    try:
        with urllib.request.urlopen(req, timeout=300) as r:
            raw = r.read()
    except urllib.error.HTTPError as e:
        raise RuntimeError('Credenziali rifiutate (401): riesegui la cella.' if e.code == 401
                           else f'Errore HTTP {e.code}.') from None
    finally:
        del password
    print(f'scaricati {len(raw):,} byte (solo in memoria, nessun file su disco)')
    if len(raw) < 100_000:
        raise RuntimeError(f'Download sospetto ({len(raw)} byte): non proseguo.')

# decodifica (resapp e' utf-8; fallback latin-1) e parsing (separatore '|')
try:
    text = raw.decode('utf-8')
except UnicodeDecodeError:
    text = raw.decode('latin-1')
del raw
df = pd.read_csv(io.StringIO(text), sep='|', dtype=str,
                 usecols=[COL_MAT, COL_COD, COL_DATA, COL_TIPO]).fillna('')
del text
print('righe totali nel dump:', len(df))

df['mat']  = df[COL_MAT].str.strip()
df['cod']  = df[COL_COD].str.strip().str.upper()
df['tipo'] = df[COL_TIPO].str.strip().str.upper()
df['data'] = pd.to_datetime(df[COL_DATA].str.strip(), format='%d-%m-%Y', errors='coerce')  # gg-mm-aaaa

n_nodate = int(df['data'].isna().sum())
print(f'date non interpretabili (scartate): {n_nodate}')   # se alto: controlla il formato di DATA_SUP
df = df[(df['mat'] != '') & (df['cod'] != '') & df['data'].notna()].copy()
print('righe con matricola+codice+data validi:', len(df))

In [ ]:
# ---- FILTRO PERIODO (per data di superamento, estremi inclusi) + solo LT/LM ----
per = df[(df['data'] >= start_d) & (df['data'] <= end_d)].copy()
per['day'] = per['data'].dt.normalize()   # giorno (per il dedup spacchettamenti)
print('esami nel periodo:', len(per), '| matricole distinte:', per['mat'].nunique())
print('  LT (L2):', int((per['tipo'] == TIPO_LT).sum()),
      '| LM:', int((per['tipo'] == TIPO_LM).sum()),
      '| altri tipi (esclusi):', int((~per['tipo'].isin([TIPO_LT, TIPO_LM])).sum()))
per = per[per['tipo'].isin([TIPO_LT, TIPO_LM])].copy()   # le coppie valgono solo entro LT o LM

In [ ]:
# ---- DEDUP SPACCHETTAMENTI ----
# Stessa matricola, STESSO GIORNO: se un codice e' PREFISSO di un altro (es. 204BB e
# 204BB_), tengo solo il piu' corto e scarto i piu' lunghi.
def _drops(codes):
    codes = set(codes)
    return {c for c in codes
            if any(o != c and len(o) < len(c) and c.startswith(o) for o in codes)}

to_drop = set()
for (m, d), codes in per.groupby(['mat', 'day'])['cod']:
    for c in _drops(codes):
        to_drop.add((m, d, c))
before = len(per)
keep = [(t.mat, t.day, t.cod) not in to_drop for t in per.itertuples(index=False)]
per = per[keep]
print(f'righe dopo dedup spacchettamenti: {len(per)} (scartate {before - len(per)})')

In [ ]:
# ---- COSTRUZIONE MATRICE (unica, normalizzata per anno, INTERI) ----
# Unita' = carriera = (matricola, CdS): le coppie si formano SOLO dentro la stessa carriera.
# I conteggi vengono divisi per ANNI e ARROTONDATI all'intero (0 -> cella vuota).
# E[code]  = carriere/anno che hanno superato il codice.
# N[(a,b)] = carriere/anno che hanno superato ENTRAMBI (a<b).
def build_matrix(sub, anni):
    per_carriera = sub.groupby(['mat', 'tipo'])['cod'].apply(lambda s: sorted(set(s)))
    E, N = Counter(), Counter()
    for codes in per_carriera:
        for c in codes:
            E[c] += 1
        for a, b in combinations(codes, 2):   # a<b (lista ordinata)
            N[(a, b)] += 1
    def norm(v):                               # conteggio -> tasso annuo intero (0 = vuoto)
        r = round(v / anni)
        return r if r else ''
    order = sorted(E, key=lambda c: (-E[c], c))    # codici per E decrescente
    pos = {c: i for i, c in enumerate(order)}
    M = [['' for _ in order] for _ in order]
    for c in order:
        M[pos[c]][pos[c]] = norm(E[c])             # diagonale = E/anno
    for (a, b), n in N.items():
        i, j = sorted((pos[a], pos[b]))
        M[i][j] = norm(n)                          # triangolo superiore = N/anno
    mat = pd.DataFrame(M, index=order, columns=order)
    return mat, E, N

In [ ]:
# ---- GENERA E SALVA (solo la matrice) ----
mat, E, N = build_matrix(per, ANNI)
out = os.path.join(OUT_DIR, f'MatriceCoesami_{START}_{END}.csv')
mat.to_csv(out, index_label='COEX')          # angolo A1 = 'COEX'; valori = tassi annui interi
n_carriere = per.groupby(['mat', 'tipo']).ngroups
print(f'matrice: {len(mat)} codici, {len(per)} esami, {n_carriere} carriere su {ANNI:.2f} anni')
top = sorted(E.items(), key=lambda kv: -kv[1])[:10]
print('top E (tot carriere | /anno):', ', '.join(f'{c}={e}|{round(e/ANNI)}' for c, e in top))
print('->', out)

In [ ]:
# ---- ANTEPRIMA (in memoria, niente salvataggio extra) ----
mat_all, _, _ = build_matrix(per, ANNI)
print('matrice', mat_all.shape, '| valori = tassi annui interi')
mat_all.iloc[:15, :15]